# Kuber — quickstart

Load a real conjugate-heat-transfer (CHT) case from the Kuber corpus and visualize the CFD target fields. **Runs on CPU in Colab — no GPU, no model weights needed.**

> Running the surrogate itself requires NVIDIA PhysicsNeMo + a checkpoint (see `checkpoints/` and the README). This notebook demonstrates the data contract and a sample case so you can see exactly what the model ingests and predicts.

In [ ]:
import os
if not os.path.exists('Kuber'):
    !git clone -q https://github.com/ShubhJain007/Kuber.git
%cd Kuber

In [ ]:
import numpy as np, json, os

# pick a sample case (3 heatsinks + 3 cold plates ship in data_sample/)
f = sorted(p for p in os.listdir('data_sample') if p.endswith('.npz'))[0]
d = np.load(f'data_sample/{f}', allow_pickle=True)

print('case      :', f)
print('arrays    :', {k: getattr(d[k], 'shape', None) for k in d.files})
print('conditions:', json.loads(str(d['conditions'])))

Each case is a 16,384-node point cloud in the fluid domain with the full 5-channel field `(U_x, U_y, U_z, T, p_rgh)` per node, plus the solid-boundary surface point cloud (`surf_pts` + `surf_normals`) that is the model's geometry input. Full contract: `docs/DATASET.md`.

In [ ]:
import matplotlib.pyplot as plt

coords = np.asarray(d['coords'])          # [N,3] metres
T = np.asarray(d['T']).reshape(-1)        # [N] Kelvin (CFD ground truth)

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='3d')
s = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=T, s=3, cmap='inferno')
ax.set_title(f'CFD temperature field — {f}')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.set_zlabel('z [m]')
fig.colorbar(s, label='T [K]', shrink=0.6)
plt.tight_layout(); plt.show()

In [ ]:
# The leaderboard + all metrics are shipped machine-readable, too:
import json
print(open('results/leaderboard.csv').read())
res = json.load(open('results/simshift_medium.json'))
print('OOD temperature RMSE (K):', res['ours']['target_out_of_distribution']['T_RMSE_K'])

## Next steps

- **Data contract**: `docs/DATASET.md`
- **Model architecture**: `docs/MODEL.md`
- **Benchmark + metrics**: `docs/BENCHMARK.md`
- **Train / evaluate** (needs PhysicsNeMo + a GPU): see the README quickstart. Evaluate a checkpoint with `python -m kuber.train_simshift --eval_only <model.pt> ...`